# Training Analysis & Rollout Visualization

This notebook pulls **all training data** from W&B, displays:
1. **Summary table** — runtime (minutes), total steps, final reward
2. **Reward curves** with ± std shading
3. **Cartesian coordinates** from a replayed rollout of a selected run

In [ ]:
import wandb
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import json
from IPython.display import display

plt.rcParams.update({
    "figure.dpi": 150,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 10,
})

## 1 — Configuration

Set your W&B entity and project. Runs are **auto-discovered** from the project.  
Use `INCLUDE_STATES` and `NAME_FILTER` to narrow down which runs to include.

In [ ]:
# ──────────────── EDIT THESE ────────────────
ENTITY  = "weissma6-zhaw-school-of-engineering"
PROJECT = "panda_pick_ppo"  # or "UR10_pick_ppo"

# Environment name used during training
ENV_NAME = "PandaPickCube"  # or "UR10PickCube"

# Filter: only include runs whose state is in this set (set to None for all)
INCLUDE_STATES = {"finished"}  # e.g. {"finished", "crashed"} or None for all

# Filter: only include runs whose name contains one of these substrings
# Set to None to include ALL runs
NAME_FILTER = None  # e.g. ["baseline", "BL_"] or None

# Rollout settings
ROLLOUT_SEED = 0
RENDER_EVERY = 1
# ────────────────────────────────────────────

## 2 — Auto-discover runs from W&B

In [ ]:
api = wandb.Api()

# ── Auto-discover all runs in the project ──
all_runs = api.runs(f"{ENTITY}/{PROJECT}", order="-created_at")
print(f"Found {len(all_runs)} total runs in {ENTITY}/{PROJECT}\n")

# Build RUNS dict: display-name → run_id (auto-dedup names)
RUNS = {}  # name → run_id
name_counts = {}
skipped = 0

for r in all_runs:
    # State filter
    if INCLUDE_STATES and r.state not in INCLUDE_STATES:
        skipped += 1
        continue
    # Name substring filter
    if NAME_FILTER is not None:
        if not any(sub.lower() in (r.name or "").lower() for sub in NAME_FILTER):
            skipped += 1
            continue

    display_name = r.name or r.id
    # De-duplicate names by appending a counter
    name_counts[display_name] = name_counts.get(display_name, 0) + 1
    if name_counts[display_name] > 1:
        display_name = f"{display_name} ({name_counts[display_name]})"
    RUNS[display_name] = r.id

print(f"Included: {len(RUNS)} runs  |  Skipped: {skipped}")
print(f"\nRuns to analyse:")
for name, rid in RUNS.items():
    print(f"  {name:60s} → {rid}")

# ── Pick the LAST finished run for the Cartesian rollout ──
ROLLOUT_RUN_ID = list(RUNS.values())[0]  # most recent
ROLLOUT_RUN_NAME = list(RUNS.keys())[0]
print(f"\nRollout run (most recent): '{ROLLOUT_RUN_NAME}' ({ROLLOUT_RUN_ID})")
print("  → Change ROLLOUT_RUN_ID above if you want a different run.")

### Fetch histories & build summary table

In [ ]:
histories = {}   # label → DataFrame
summaries = []   # list of dicts for the summary table

for label, run_id in RUNS.items():
    run = api.run(f"{ENTITY}/{PROJECT}/{run_id}")

    # ── History (reward curves) ──
    hist = run.history(
        keys=["eval/episode_reward", "eval/episode_reward_std", "training/num_steps"],
        pandas=True,
    ).dropna(subset=["eval/episode_reward"])
    histories[label] = hist

    # ── Run metadata ──
    runtime_min = run.summary.get("_runtime", 0) / 60.0

    total_steps = (
        int(hist["training/num_steps"].max())
        if "training/num_steps" in hist.columns and len(hist) > 0
        else int(run.summary.get("training/num_steps", 0))
    )
    final_reward = (
        float(hist["eval/episode_reward"].iloc[-1])
        if len(hist) > 0
        else float(run.summary.get("eval/episode_reward", float("nan")))
    )

    summaries.append({
        "Run":            label,
        "ID":             run_id,
        "State":          run.state,
        "Runtime (min)":  round(runtime_min, 2),
        "Total Steps":    total_steps,
        "Final Reward":   round(final_reward, 2),
    })

    print(f"✓ {label:50s}  steps={total_steps:>12,}  runtime={runtime_min:6.1f} min  reward={final_reward:.2f}")

summary_df = pd.DataFrame(summaries)
display(summary_df)

## 3 — Reward curves (mean ± std)

In [ ]:
n = len(histories)
colors = plt.cm.tab10.colors[:n]

fig, axes = plt.subplots(1, n, figsize=(5 * n, 4), sharey=True, squeeze=False)
axes = axes.flatten()

for ax, (label, df), color in zip(axes, histories.items(), colors):
    steps = df["_step"] if "_step" in df.columns else df.index
    mean  = df["eval/episode_reward"]
    std   = df.get("eval/episode_reward_std", pd.Series(0, index=df.index))

    ax.plot(steps, mean, color=color, linewidth=2, label="Mean Reward")
    ax.fill_between(steps, mean - std, mean + std, color=color, alpha=0.2, label="± Std Dev")

    ax.set_xlabel("Environment Steps")
    ax.set_title(label, fontweight="bold")
    ax.legend(loc="lower right", fontsize=8)
    ax.xaxis.set_major_formatter(
        plt.FuncFormatter(lambda x, _: f"{x/1e6:.1f}M" if x >= 1e6 else f"{x/1e3:.0f}K")
    )

axes[0].set_ylabel("Episode Reward")
plt.suptitle("Training Reward Curves", fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("reward_comparison.png", bbox_inches="tight", dpi=300)
plt.savefig("reward_comparison.pdf", bbox_inches="tight", dpi=300)
plt.show()

## 4 — All-runs overlay (single plot)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))

for (label, df), color in zip(histories.items(), colors):
    steps = df["_step"] if "_step" in df.columns else df.index
    mean  = df["eval/episode_reward"]
    std   = df.get("eval/episode_reward_std", pd.Series(0, index=df.index))

    ax.plot(steps, mean, color=color, linewidth=2, label=label)
    ax.fill_between(steps, mean - std, mean + std, color=color, alpha=0.15)

ax.set_xlabel("Environment Steps")
ax.set_ylabel("Episode Reward")
ax.set_title("All Runs — Reward Overlay", fontweight="bold")
ax.legend(fontsize=9)
ax.xaxis.set_major_formatter(
    plt.FuncFormatter(lambda x, _: f"{x/1e6:.1f}M" if x >= 1e6 else f"{x/1e3:.0f}K")
)
plt.tight_layout()
plt.savefig("reward_overlay.png", bbox_inches="tight", dpi=300)
plt.show()

## 5 — Runtime & Steps bar charts

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))

labels = summary_df["Run"]
x = np.arange(len(labels))

# Runtime
ax1.barh(x, summary_df["Runtime (min)"], color="steelblue")
ax1.set_yticks(x)
ax1.set_yticklabels(labels)
ax1.set_xlabel("Runtime (minutes)")
ax1.set_title("Training Runtime", fontweight="bold")
for i, v in enumerate(summary_df["Runtime (min)"]):
    ax1.text(v + 0.3, i, f"{v:.1f}", va="center", fontsize=9)

# Steps
ax2.barh(x, summary_df["Total Steps"], color="coral")
ax2.set_yticks(x)
ax2.set_yticklabels(labels)
ax2.set_xlabel("Total Training Steps")
ax2.set_title("Training Steps", fontweight="bold")
ax2.xaxis.set_major_formatter(
    plt.FuncFormatter(lambda x, _: f"{x/1e6:.1f}M" if x >= 1e6 else f"{x/1e3:.0f}K")
)

plt.tight_layout()
plt.savefig("runtime_steps.png", bbox_inches="tight", dpi=300)
plt.show()

---
## 6 — Cartesian Coordinates & Video from W&B

Pulls the **rollout coordinate table** and **last video** directly from the W&B run.  
*(Requires that the run used the updated `UR10_ppo.py` which logs `eval/rollout_coords`.  
For older runs without the table, a CSV fallback is attempted from run files.)*

In [ ]:
import os, io, tempfile
from IPython.display import Video, display, HTML

rollout_run = api.run(f"{ENTITY}/{PROJECT}/{ROLLOUT_RUN_ID}")
print(f"Rollout run: '{rollout_run.name}' ({ROLLOUT_RUN_ID})")
print(f"State: {rollout_run.state}  |  Runtime: {rollout_run.summary.get('_runtime',0)/60:.1f} min\n")

# ── List all files in the run ──
run_files = list(rollout_run.files())
print(f"Run has {len(run_files)} files:")
for f in run_files:
    print(f"  {f.name:60s}  ({f.size:>10,} bytes)")

### 6a — Download & display the last evaluation video

In [ ]:
# Find video files (mp4) in the run
video_files = [f for f in run_files if f.name.endswith(".mp4")]

if not video_files:
    # Also check media/videos path
    video_files = [f for f in run_files if "video" in f.name.lower() and (
        f.name.endswith(".mp4") or f.name.endswith(".webm") or f.name.endswith(".gif")
    )]

if video_files:
    # Take the LAST video (highest training step)
    last_video = video_files[-1]
    print(f"Last video: {last_video.name}  ({last_video.size:,} bytes)")

    os.makedirs("downloaded_videos", exist_ok=True)
    last_video.download(root="downloaded_videos", replace=True)
    video_local = os.path.join("downloaded_videos", last_video.name)
    print(f"Downloaded to: {video_local}")

    # Display inline
    display(Video(video_local, embed=True, width=600))
else:
    print("⚠ No video files found in this run.")
    print("  Check W&B UI: run → Files → media/videos/eval")

### 6b — Fetch rollout data (ctrl, body positions, torques)

Looks for a `.csv` with the same base name as the video file.  
Falls back to `rollout_coords_*.csv` from the earlier version.

In [ ]:
rollout_df = None

# ── Strategy 1: CSV matching the video filename (new format — has ctrl + torques) ──
# Video is named e.g. EnvName_runid_rew123.4_steps1000000.mp4
# CSV uses the same base name with .csv extension
if video_files:
    video_base = os.path.splitext(video_files[-1].name)[0]  # same base as last video
    matching_csvs = [f for f in run_files if f.name == video_base + '.csv']
    if not matching_csvs:
        # Fallback: any non-rollout_coords CSV
        matching_csvs = [f for f in run_files if f.name.endswith('.csv') and 'rollout_coords' not in f.name]
    if matching_csvs:
        target = matching_csvs[-1]
        print(f'Found CSV: {target.name}  ({target.size:,} bytes)')
        os.makedirs('downloaded_data', exist_ok=True)
        target.download(root='downloaded_data', replace=True)
        rollout_df = pd.read_csv(os.path.join('downloaded_data', target.name))
        print(f'✓ Loaded {len(rollout_df)} rows, {len(rollout_df.columns)} columns')

# ── Strategy 2: rollout_coords CSV (old format — body positions only) ──
if rollout_df is None:
    coord_csvs = [f for f in run_files if f.name.startswith('rollout_coords') and f.name.endswith('.csv')]
    if coord_csvs:
        target = coord_csvs[-1]
        print(f'Found rollout_coords CSV: {target.name}  ({target.size:,} bytes)')
        os.makedirs('downloaded_data', exist_ok=True)
        target.download(root='downloaded_data', replace=True)
        rollout_df = pd.read_csv(os.path.join('downloaded_data', target.name))
        print(f'✓ Loaded {len(rollout_df)} rows (old format, no ctrl/torques)')

if rollout_df is not None:
    print(f'\nColumns ({len(rollout_df.columns)}):')
    for c in rollout_df.columns:
        print(f'  {c}')
    display(rollout_df.head(5))
else:
    print('\n⚠ No rollout data found for this run.')
    print('  Re-run training with the updated UR10_ppo.py to log rollout data.')


### 6c — Identify available data channels

In [ ]:
assert rollout_df is not None, "No rollout data — see instructions above."

cols = list(rollout_df.columns)

# ── Detect ctrl columns ──
ctrl_cols = [c for c in cols if c.startswith("ctrl_")]
print(f"Control columns ({len(ctrl_cols)}): {ctrl_cols}")

# ── Detect body position columns (pattern: <name>_x, <name>_y, <name>_z) ──
body_xyz_sets = {}
for c in cols:
    if c.endswith('_x'):
        base = c[:-2]
        if f"{base}_y" in cols and f"{base}_z" in cols and base not in ('ctrl', 'torque'):
            body_xyz_sets[base] = (f"{base}_x", f"{base}_y", f"{base}_z")
print(f"\nBody position sets ({len(body_xyz_sets)}):")
for name in body_xyz_sets:
    print(f"  {name}")

# ── Detect torque columns ──
torque_cols = [c for c in cols if c.startswith("torque_")]
print(f"\nTorque columns ({len(torque_cols)}): {torque_cols}")

In [ ]:
# ──────────── EDIT THESE to match your model ────────────
# Pick body names from the list printed above
body_list = list(body_xyz_sets.keys())
EE_BODY   = body_list[-2] if len(body_list) >= 2 else body_list[-1]  # end-effector
CUBE_BODY = body_list[-1]                                             # cube / object
# ────────────────────────────────────────────────────────

print(f"End-effector body: '{EE_BODY}'")
print(f"Object body:       '{CUBE_BODY}'")

## 7 — Plot Control Signals

In [ ]:
if ctrl_cols:
    n_ctrl = len(ctrl_cols)
    fig, axes = plt.subplots(n_ctrl, 1, figsize=(10, 2.2 * n_ctrl), sharex=True)
    if n_ctrl == 1:
        axes = [axes]
    for ax, col in zip(axes, ctrl_cols):
        ax.plot(rollout_df["timestep"], rollout_df[col], linewidth=1)
        ax.set_ylabel(col, fontsize=8)
    axes[-1].set_xlabel("Timestep")
    axes[0].set_title(f"Control Signals — {rollout_run.name}", fontweight="bold")
    plt.tight_layout()
    plt.savefig("ctrl_signals.png", bbox_inches="tight", dpi=300)
    plt.show()
else:
    print("No ctrl columns found (old CSV format).")

## 8 — Cartesian Trajectories

In [ ]:
ee_x, ee_y, ee_z    = body_xyz_sets[EE_BODY]
cb_x, cb_y, cb_z    = body_xyz_sets[CUBE_BODY]

fig, axes = plt.subplots(3, 1, figsize=(10, 8), sharex=True)
for ax, coord, ee_c, cb_c in zip(axes, ["X","Y","Z"],
                                   [ee_x, ee_y, ee_z],
                                   [cb_x, cb_y, cb_z]):
    ax.plot(rollout_df["timestep"], rollout_df[ee_c], lw=1.5, label=f"{EE_BODY} {coord}")
    ax.plot(rollout_df["timestep"], rollout_df[cb_c], lw=1.5, ls="--", label=f"{CUBE_BODY} {coord}")
    ax.set_ylabel(f"{coord} (m)")
    ax.legend(loc="upper right", fontsize=8)

axes[-1].set_xlabel("Timestep")
axes[0].set_title(f"Cartesian Coordinates — {rollout_run.name}", fontweight="bold")
plt.tight_layout()
plt.savefig("cartesian_coords_xyz.png", bbox_inches="tight", dpi=300)
plt.show()

In [ ]:
# ── 3D trajectory ──
fig = plt.figure(figsize=(8, 8))
ax = fig.add_subplot(111, projection="3d")

ax.plot(rollout_df[ee_x], rollout_df[ee_y], rollout_df[ee_z],
        lw=1.5, label=EE_BODY, color="tab:blue")
ax.plot(rollout_df[cb_x], rollout_df[cb_y], rollout_df[cb_z],
        lw=1.5, label=CUBE_BODY, color="tab:orange", ls="--")

ax.scatter(*[rollout_df[c].iloc[0]  for c in [ee_x, ee_y, ee_z]], s=80, c="green", marker="o", label="EE start")
ax.scatter(*[rollout_df[c].iloc[-1] for c in [ee_x, ee_y, ee_z]], s=80, c="red",   marker="X", label="EE end")
ax.scatter(*[rollout_df[c].iloc[0]  for c in [cb_x, cb_y, cb_z]], s=80, c="green", marker="s")
ax.scatter(*[rollout_df[c].iloc[-1] for c in [cb_x, cb_y, cb_z]], s=80, c="red",   marker="D")

ax.set_xlabel("X (m)"); ax.set_ylabel("Y (m)"); ax.set_zlabel("Z (m)")
ax.set_title(f"3D Trajectory — {rollout_run.name}", fontweight="bold")
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig("cartesian_3d_trajectory.png", bbox_inches="tight", dpi=300)
plt.show()

In [ ]:
# ── EE ↔ Cube distance ──
dist = np.sqrt(
    (rollout_df[ee_x].values - rollout_df[cb_x].values)**2 +
    (rollout_df[ee_y].values - rollout_df[cb_y].values)**2 +
    (rollout_df[ee_z].values - rollout_df[cb_z].values)**2
)

fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(rollout_df["timestep"], dist, color="tab:purple", lw=1.5)
ax.set_xlabel("Timestep"); ax.set_ylabel("Distance (m)")
ax.set_title(f"EE ↔ Object Distance — {rollout_run.name}", fontweight="bold")
ax.axhline(0, color="gray", ls=":", alpha=0.5)
plt.tight_layout()
plt.savefig("ee_cube_distance.png", bbox_inches="tight", dpi=300)
plt.show()

print(f"Min distance: {dist.min():.4f} m  at step {dist.argmin()}")
print(f"Final distance: {dist[-1]:.4f} m")

## 9 — Actuator Torques

In [ ]:
if torque_cols:
    n_torq = len(torque_cols)
    fig, axes = plt.subplots(n_torq, 1, figsize=(10, 2.2 * n_torq), sharex=True)
    if n_torq == 1:
        axes = [axes]
    for ax, col in zip(axes, torque_cols):
        ax.plot(rollout_df["timestep"], rollout_df[col], linewidth=1, color="tab:red")
        ax.set_ylabel(col.replace('torque_',''), fontsize=8)
    axes[-1].set_xlabel("Timestep")
    axes[0].set_title(f"Actuator Torques — {rollout_run.name}", fontweight="bold")
    plt.tight_layout()
    plt.savefig("torques.png", bbox_inches="tight", dpi=300)
    plt.show()

    # Summary stats
    torque_summary = rollout_df[torque_cols].describe().T
    torque_summary.columns = [c.capitalize() for c in torque_summary.columns]
    display(torque_summary)
else:
    print("No torque columns found.")
    print("Torques are logged by the updated UR10_ppo.py — re-run training to get them.")

## 10 — Export all data

In [ ]:
rollout_df.to_csv("rollout_full_export.csv", index=False)
print(f"Saved rollout_full_export.csv  ({len(rollout_df)} rows × {len(rollout_df.columns)} cols)")
display(rollout_df.describe().T)